# Pipeline Validation Checks

Este notebook se utiliza para realizar verificaciones rápidas del pipeline y validar que la base de datos contiene los resultados esperados.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///lsg50.db")

In [16]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", engine)

,name
0,fundamentals_snapshot
1,sqlite_sequence
2,lsg50_composition
3,lsg50_index_members
4,universe


In [17]:
composition = pd.read_sql("""
SELECT *
FROM lsg50_composition
WHERE calculation_date = (
    SELECT MAX(calculation_date)
    FROM lsg50_composition
)
""", engine)

composition = composition.drop_duplicates(subset="symbol")

composition.head()

,id,symbol,calculation_date,weight,cfo_index
0,1,NVDA,2026-03-05,0.023272,0.998874
1,2,GE,2026-03-05,0.022590,0.969595
2,3,LLY,2026-03-05,0.022302,0.957207
3,4,META,2026-03-05,0.022065,0.947072
4,5,TMUS,2026-03-05,0.021829,0.936937


In [18]:
composition["weight"].sum()

np.float64(1.0)

In [19]:
top10 = composition.sort_values("weight", ascending=False).head(10)

top10

,id,symbol,calculation_date,weight,cfo_index
0,1,NVDA,2026-03-05,0.023272,0.998874
1,2,GE,2026-03-05,0.022590,0.969595
2,3,LLY,2026-03-05,0.022302,0.957207
3,4,META,2026-03-05,0.022065,0.947072
4,5,TMUS,2026-03-05,0.021829,0.936937
5,6,AMD,2026-03-05,0.021803,0.935811
6,7,GOOG,2026-03-05,0.021711,0.931869
7,8,GOOGL,2026-03-05,0.021685,0.930743
8,9,NFLX,2026-03-05,0.021488,0.922297
9,10,CRM,2026-03-05,0.021462,0.921171


In [20]:
sector_data = pd.read_sql("""
SELECT symbol, sector
FROM universe
""", engine)

composition_sector = composition.merge(sector_data, on="symbol")

composition_sector.head()

,id,symbol,calculation_date,weight,cfo_index,sector
0,1,NVDA,2026-03-05,0.023272,0.998874,Information Technology
1,2,GE,2026-03-05,0.022590,0.969595,Industrials
2,3,LLY,2026-03-05,0.022302,0.957207,Health Care
3,4,META,2026-03-05,0.022065,0.947072,Communication Services
4,5,TMUS,2026-03-05,0.021829,0.936937,Communication Services


In [21]:
sector_weights = (
    composition_sector
    .groupby("sector")["weight"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

sector_weights

,sector,weight
0,Information Technology,0.226006
1,Financials,0.173296
2,Communication Services,0.150338
3,Health Care,0.121609
4,Industrials,0.099675
5,Consumer Staples,0.077216
6,Real Estate,0.057827
7,Consumer Discretionary,0.037047
8,Materials,0.019494
9,Utilities,0.019232


In [23]:
composition_sector.to_csv(
    "../tableau/lsg50_composition_latest.csv",
    index=False
)

sector_weights.to_csv(
    "../tableau/lsg50_sector_weights.csv",
    index=False
)

top10.to_csv(
    "../tableau/lsg50_top10_holdings.csv",
    index=False
)

print("Datasets exported for Tableau.")

Datasets exported for Tableau.
